In [1]:
import numpy as np
import random
from models import Model
import torch
import time
import pandas as pd

import pynvml
import threading

/workspace/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:68: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
from equations import flops, memory, latency, energy 

In [3]:
model = Model()

In [4]:
torch.backends.cudnn.benchmark = False        # main results: PyTorch's default heuristics choose the kernel
torch.backends.cudnn.allow_tf32 = False       # so that FP32 means FP32 on Ampere and newer GPUs
torch.backends.cuda.matmul.allow_tf32 = False
model = model.cuda().eval()

In [5]:
random.seed(42)

In [6]:
S = [32, 64, 128, 224, 256, 384, 512, 1024]
B = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]

grid = list(range(32, 512, 16))
cands = [size for size in grid if size not in S]
S.extend(random.sample(cands, 4))

power2 = [2**i for i in range(1, 9)]
b_grid = list(range(1, 257))
b_cands = [b for b in b_grid if b not in power2]
B.extend(random.sample(b_cands, 3))

# S = [32, 512]
# B = [16, 128]
S = np.array(S)
B = np.array(B)

S = S[:, None]
B = B[None, :]

In [7]:
def generate_random_batch(image_size: int, batch: int, device='cuda'):
    return torch.randn(batch, 3, image_size, image_size, device=device, dtype=torch.float32)

In [8]:
flops_results = flops(S, B) 
memory_results = memory(S, B)

In [9]:
def measure_latency(model, x, n_warmup: int = 10, n_repeats: int = 30):
    model.eval()
    with torch.inference_mode():
        for _ in range(n_warmup):
            _ = model(x)
        torch.cuda.synchronize()
        times = []
        for _ in range(n_repeats):
            torch.cuda.synchronize()
            start = time.perf_counter()
            _ = model(x)
            torch.cuda.synchronize()
            times.append(time.perf_counter() - start)
        return float(np.median(times))        

In [10]:
pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)

def get_power_watts():
    return pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0


def measure_energy(model, x, n_warmup=10, n_repeats=10, sample_interval=0.001):
    model.eval()

    with torch.inference_mode():
        for _ in range(n_warmup):
            _ = model(x)
        torch.cuda.synchronize()

        energies = []
        for _ in range(n_repeats):
            power_samples = []
            timestamps = []
            stop_flag = threading.Event()

            def sampler():
                while not stop_flag.is_set():
                    power_samples.append(get_power_watts())
                    timestamps.append(time.perf_counter())
                    time.sleep(sample_interval)

            thread = threading.Thread(target=sampler)

            torch.cuda.synchronize()
            thread.start()
            start = time.perf_counter()

            _ = model(x)
            torch.cuda.synchronize()

            end = time.perf_counter()
            stop_flag.set()
            thread.join()

            if len(power_samples) >= 2:
                energy_j = np.trapezoid(power_samples, timestamps)
            else:
                energy_j = np.mean(power_samples) * (end - start) if power_samples else 0.0

            energies.append(energy_j)

    return float(np.median(energies))

def measure_energy_batched(model, x, n_forward_passes=50, sample_interval=0.01):
    power_samples = []
    timestamps = []
    stop_flag = threading.Event()

    def sampler():
        while not stop_flag.is_set():
            power_samples.append(get_power_watts())
            timestamps.append(time.perf_counter())
            time.sleep(sample_interval)

    thread = threading.Thread(target=sampler)

    with torch.inference_mode():
        torch.cuda.synchronize()
        thread.start()
        for _ in range(n_forward_passes):
            _ = model(x)
        torch.cuda.synchronize()
        stop_flag.set()
        thread.join()

    total_energy = np.trapezoid(power_samples, timestamps)
    energy_per_forward = total_energy / n_forward_passes
    return energy_per_forward

In [11]:
result = []
for i, s in enumerate(S[:, 0]):
    for j, b in enumerate(B[0]):
        # print(i, j, s, b, memory_results[i, j])
        s, b = int(s), int(b)
        try:
            x = generate_random_batch(s, b, device='cuda')
            latency_measured = measure_latency(model, x)
            energy_measured = measure_energy_batched(model, x)
        
            torch.cuda.reset_peak_memory_stats()
            with torch.inference_mode():
                _ = model(x)
            result.append({
                'S': s, 'B': b, 
                'latency': latency_measured, 
                'memory, mb': np.round(torch.cuda.max_memory_allocated() / (1024 ** 2), 2), 
                'energy': energy_measured,
                'memory_calculated, mb': np.round(memory_results[i, j] / (1024 ** 2), 2)
            
            })
            del x
            
        except torch.cuda.OutOfMemoryError:
            result.append({
                'S': s, 'B': b, 
                'latency': None, 'memory, mb': 'OOM', 'energy': None, 
                'memory_calculated, mb': np.round(memory_results[i, j] / (1024 ** 2), 2)
            })
        torch.cuda.empty_cache()

[W925 09:47:56.250688200 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 2147483648 bytes (free: 501350400, total: 12485525504).
[W925 09:47:56.250920242 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 2147483648 bytes (free: 501350400, total: 12485525504).
[W925 09:47:56.695433411 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 17179869184 bytes (free: 2648834048, total: 12485525504).
[W925 09:47:56.804022904 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 17179869184 bytes (free: 5870059520, total: 12485525504).


In [12]:
res = pd.DataFrame(result)
res.sort_values(by='memory_calculated, mb', ascending=False)

,S,B,latency,"memory, mb",energy,"memory_calculated, mb"
100,1024,512,NaN,OOM,NaN,53251.97
99,1024,256,NaN,OOM,NaN,26627.97
98,1024,128,0.465998,8716.12,72.889296,13315.97
87,512,512,0.463188,8716.19,71.660098,13315.97
152,496,512,0.437379,8182.19,68.584219,12496.97
...,...,...,...,...,...,...
2,32,4,0.000626,12.36,0.013695,4.37
13,64,1,0.000640,12.38,0.025335,4.37
130,48,1,0.000779,12.27,0.087779,4.20
1,32,2,0.000634,12.23,0.013635,4.17


In [13]:
res.to_csv("mesaruments.csv", index=False)

### Калибровка

In [14]:
df = pd.read_csv("mesaruments.csv")
df.head()

,S,B,latency,"memory, mb",energy,"memory_calculated, mb"
0,32,1,0.000642,12.16,0.013620,4.07
1,32,2,0.000634,12.23,0.013635,4.17
2,32,4,0.000626,12.36,0.013695,4.37
3,32,8,0.000666,12.63,0.013637,4.78
4,32,16,0.000689,18.48,0.020490,5.59


In [15]:
from scipy.optimize import curve_fit

In [16]:
calib = df[df['memory, mb'] != 'OOM']
calib

,S,B,latency,"memory, mb",energy,"memory_calculated, mb"
0,32,1,0.000642,12.16,0.013620,4.07
1,32,2,0.000634,12.23,0.013635,4.17
2,32,4,0.000626,12.36,0.013695,4.37
3,32,8,0.000666,12.63,0.013637,4.78
4,32,16,0.000689,18.48,0.020490,5.59
...,...,...,...,...,...,...
151,496,256,0.219255,4097.39,34.615416,6250.47
152,496,512,0.437379,8182.19,68.584219,12496.97
153,496,77,0.066934,1241.7,10.483593,1882.80
154,496,69,0.060229,1114.43,9.436444,1687.60


In [17]:
S_data = calib['S'].values.astype(float)
B_data = calib['B'].values.astype(float)
lat_data = calib["latency"].values

In [18]:
def latency_wrapper(X, theta0, theta1, theta2):
    S, B = X
    return latency(S, B, [theta0, theta1, theta2])

In [19]:
p0 = [1e-3, 1e-13, 1e-11]

theta_lat, _ = curve_fit(
    latency_wrapper,
    (S_data, B_data),
    lat_data,
    p0=p0,
    bounds=([0, 0, 0], [1, 1e-9, 1e-7]),
    maxfev=20000
)
print("theta0 (launch overhead, s):", theta_lat[0])
print("theta1 (s/FLOP):", theta_lat[1])
print("theta2 (s/byte):", theta_lat[2])

theta0 (launch overhead, s): 0.0007358872498243746
theta1 (s/FLOP): 1.9530910897041277e-13
theta2 (s/byte): 1.9528532137814407e-13


In [20]:
energy_data = calib['energy'].values

def energy_wrapper(X, theta_s, theta_f, theta_b):
    S, B = X
    return energy(S, B, theta_lat, [theta_s, theta_f, theta_b])

p0_energy = [70.0, 1e-10, 1e-9]

theta_en, _ = curve_fit(
    energy_wrapper,
    (S_data, B_data),
    energy_data,
    p0=p0_energy,
    bounds=([0, 0, 0], [500, 1e-6, 1e-6]),
    maxfev=20000
)
print("theta_s (W, static power):", theta_en[0])
print("theta_f (J/FLOP):", theta_en[1])
print("theta_b (J/byte):", theta_en[2])

theta_s (W, static power): 1.8600716672379595e-11
theta_f (J/FLOP): 2.960788643620542e-11
theta_b (J/byte): 1.59812620033682e-10
